# Day 21. 전체 구조 복습

**오늘의 목표**
- 지난 3주 동안 배운 조각들이 **하나의 GPT 코드 안에서 어디에 있는지** 지도처럼 본다
- `"나는 오늘 학교에 간다"`가 모델을 통과하는 동안 **shape가 어떻게 바뀌는지** 끝까지 따라간다
- 각 공식(`y = xW + b`, `softmax(QKᵀ/√dk)V`, LayerNorm, Cross Entropy)이 코드의 **몇 번째 줄**인지 찾는다
- 짧게 학습시켜서 Loss가 `log(V)`에서 내려가는 것을 확인한다
- 커리큘럼의 **최종 질문 전체**에 스스로 답해본다

> 오늘은 새로운 개념이 없습니다. 지금까지 배운 것을 **한 장의 지도**로 묶는 날이에요.
> 4주차에서는 이 코드를 **처음부터 한 줄씩 직접** 다시 만들고 제대로 학습시킵니다.

In [ ]:
import math
import inspect
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()]

## 1. mini GPT 전체 코드 (지도)

아래 셀 하나가 **GPT 전체**입니다. 모든 줄 옆에 **어느 Day에서 배웠는지** 적어두었어요.
처음 보면 길어 보이지만, 한 줄 한 줄은 전부 이미 해본 것입니다.

| 부품 | 하는 일 | 배운 Day |
|---|---|---|
| Tokenizer | 문장 ↔ Token ID | Day 8 |
| `CausalSelfAttention` | Token끼리 정보 교환 (미래는 못 봄) | Day 11~14 |
| `MLP` (FFN) | Token마다 따로 생각하기 | Day 4, 17 |
| `Block` | Attention + FFN + Residual + LayerNorm | Day 15, 16 |
| `GPT` | Embedding → Block × N → 점수 → Loss / 생성 | Day 9, 10, 19, 20 |

### 1-1. Tokenizer와 데이터 (Day 8)

In [ ]:
text = (
    "나는 오늘 학교에 간다. 나는 어제 도서관에 갔다. 오늘 날씨가 좋다. "
    "내일은 날씨가 춥다. 친구는 오늘 학교에 간다. 나는 학교에서 공부를 한다. "
    "친구는 도서관에서 책을 읽는다. 오늘은 수업이 많다. 나는 집에 간다. "
    "철수는 학교에 갔다. 그는 공부를 했다. 영희는 도서관에 갔다. 그녀는 책을 읽었다. "
)

chars = sorted(set(text))                       # Day 8: Vocabulary = 등장한 글자 전부
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}    # Day 8: 글자 → ID
itos = {i: ch for i, ch in enumerate(chars)}    # Day 8: ID → 글자
encode = lambda s: [stoi[c] for c in s]         # Day 8: 문장 → ID 리스트
decode = lambda ids: "".join(itos[i] for i in ids)  # Day 8: ID 리스트 → 문장

data = torch.tensor(encode(text), dtype=torch.long)
print("vocab_size :", vocab_size)
print("전체 Token 수:", len(data))
print("encode('나는 오늘'):", encode("나는 오늘"))
print("decode 되돌리기 :", decode(encode("나는 오늘")))

### 1-2. 모델 (Day 3~20 전부)

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, C, n_head, block_size):
        super().__init__()
        assert C % n_head == 0                                  # Day 14: C가 head 개수로 나눠떨어져야 함
        self.n_head = n_head
        self.c_attn = nn.Linear(C, 3 * C)                       # Day 12: Wq, Wk, Wv를 한 번에 (Day 14 구현 B)
        self.c_proj = nn.Linear(C, C)                           # Day 14: Head를 합친 뒤 섞는 Linear
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))  # Day 14: Causal Mask

    def forward(self, x):
        B, T, C = x.shape                                       # Day 1: (B, T, C)
        q, k, v = self.c_attn(x).split(C, dim=2)                # Day 12: Q, K, V 각각 (B, T, C)
        hs = C // self.n_head                                   # Day 14: head_size
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)       # Day 14: (B, n_head, T, head_size)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)       # Day 14
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)       # Day 14
        att = q @ k.transpose(-2, -1) / math.sqrt(hs)           # Day 2·12: QK^T,  Day 13: / √dk
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf"))  # Day 5·14: 미래는 -inf
        att = F.softmax(att, dim=-1)                            # Day 5·11: 행마다 합이 1인 Weight
        y = att @ v                                             # Day 11·12: V를 Weight대로 섞기
        y = y.transpose(1, 2).contiguous().view(B, T, C)        # Day 14: Head 다시 합치기
        return self.c_proj(y)                                   # Day 3·14: y = xW + b


class MLP(nn.Module):                                           # Day 17: Feed Forward Network
    def __init__(self, C):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(C, 4 * C),                                # Day 3·17: 4배로 늘리기
            nn.GELU(),                                          # Day 4: 비선형성
            nn.Linear(4 * C, C),                                # Day 3·17: 원래 크기로
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):                                         # Day 15: Transformer Block
    def __init__(self, C, n_head, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(C)                              # Day 16: LayerNorm
        self.attn = CausalSelfAttention(C, n_head, block_size)  # Day 11~14
        self.ln2 = nn.LayerNorm(C)                              # Day 16
        self.mlp = MLP(C)                                       # Day 17

    def forward(self, x):
        x = x + self.attn(self.ln1(x))                          # Day 15·16: Pre-LN + Residual
        x = x + self.mlp(self.ln2(x))                           # Day 15·16: Pre-LN + Residual
        return x


class GPT(nn.Module):                                           # Day 19: GPT 구조
    def __init__(self, vocab_size, C, n_head, n_layer, block_size):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, C)              # Day 9: Token Embedding
        self.pos_emb = nn.Embedding(block_size, C)              # Day 10: Position Embedding
        self.blocks = nn.Sequential(*[Block(C, n_head, block_size) for _ in range(n_layer)])  # Day 15: Block × N
        self.ln_f = nn.LayerNorm(C)                             # Day 16·19: 마지막 LayerNorm
        self.lm_head = nn.Linear(C, vocab_size, bias=False)     # Day 3·19: 벡터 → Vocabulary 점수

    def forward(self, idx, targets=None):
        B, T = idx.shape                                        # Day 8: Token ID (B, T)
        tok = self.tok_emb(idx)                                 # Day 9: (B, T, C)
        pos = self.pos_emb(torch.arange(T, device=idx.device))  # Day 10: (T, C)
        x = tok + pos                                           # Day 10: Broadcasting으로 더하기
        x = self.blocks(x)                                      # Day 15: (B, T, C) 그대로
        x = self.ln_f(x)                                        # Day 16
        logits = self.lm_head(x)                                # Day 19: (B, T, vocab_size)

        loss = None
        if targets is not None:                                 # Day 20: 학습할 때만
            V = logits.size(-1)
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))  # Day 6: Cross Entropy
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):  # Day 20: Autoregressive 생성
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]                # Day 10·14: block_size까지만 볼 수 있음
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature             # Day 1: 마지막 위치, Day 5: Temperature
            probs = F.softmax(logits, dim=-1)                   # Day 5: 점수 → 확률
            next_id = torch.multinomial(probs, num_samples=1)   # Day 5: Sampling
            idx = torch.cat([idx, next_id], dim=1)              # Day 20: 뒤에 붙이고 반복
        return idx

In [ ]:
C, n_head, n_layer, block_size = 64, 4, 2, 32

model = GPT(vocab_size, C, n_head, n_layer, block_size)
n_params = sum(p.numel() for p in model.parameters())
print(f"설정: vocab_size={vocab_size}, C={C}, n_head={n_head}, n_layer={n_layer}, block_size={block_size}")
print(f"파라미터 개수: {n_params:,}")

## 2. `"나는 오늘 학교에 간다"`의 여행: shape 추적

모델 안의 부품을 **하나씩 직접 호출**하면서 shape가 어떻게 바뀌는지 따라가봅시다.
(커리큘럼 학습 원칙 3번: "직접 값을 출력한다")

In [ ]:
rows = []
def log(step, t, day):
    rows.append((step, tuple(t.shape), day))

sentence = "나는 오늘 학교에 간다"
ids = torch.tensor([encode(sentence)])          # (1, T)
B, T = ids.shape
log("Token ID", ids, "Day 8")

with torch.no_grad():
    tok = model.tok_emb(ids);                         log("Token Embedding", tok, "Day 9")
    pos = model.pos_emb(torch.arange(T));             log("Position Embedding", pos, "Day 10")
    x = tok + pos;                                    log("tok + pos", x, "Day 10")

    for i, blk in enumerate(model.blocks):
        h = blk.ln1(x);                               log(f"[Block {i}] ln1", h, "Day 16")
        q, k, v = blk.attn.c_attn(h).split(C, dim=2); log(f"[Block {i}] q (나누기 전)", q, "Day 12")
        hs = C // n_head
        q4 = q.view(B, T, n_head, hs).transpose(1, 2); log(f"[Block {i}] q (head 나눔)", q4, "Day 14")
        k4 = k.view(B, T, n_head, hs).transpose(1, 2)
        att = q4 @ k4.transpose(-2, -1) / math.sqrt(hs); log(f"[Block {i}] QK^T / √dk", att, "Day 12·13")
        a = blk.attn(h);                              log(f"[Block {i}] Attention 출력", a, "Day 14")
        x = x + a;                                    log(f"[Block {i}] + Residual", x, "Day 16")
        h2 = blk.ln2(x);                              log(f"[Block {i}] ln2", h2, "Day 16")
        up = blk.mlp.net[0](h2);                      log(f"[Block {i}] FFN 확장 (×4)", up, "Day 17")
        m = blk.mlp(h2);                              log(f"[Block {i}] FFN 출력", m, "Day 17")
        x = x + m;                                    log(f"[Block {i}] + Residual", x, "Day 16")

    x = model.ln_f(x);                                log("ln_f", x, "Day 16·19")
    logits = model.lm_head(x);                        log("logits", logits, "Day 19")
    last = logits[:, -1, :];                          log("마지막 위치 logits", last, "Day 20")
    probs = F.softmax(last, dim=-1);                  log("Softmax 확률", probs, "Day 5")
    next_id = torch.multinomial(probs, 1);            log("다음 Token ID", next_id, "Day 5·20")

print(f"{'단계':<26}{'shape':<20}배운 Day")
print("-" * 58)
for step, shape, day in rows:
    print(f"{step:<26}{str(shape):<20}{day}")
print()
print(f"T = {T} ('{sentence}'는 글자 {T}개, 띄어쓰기 포함)")
print("다음 글자 (아직 학습 전이라 랜덤):", repr(decode(next_id[0].tolist())))

**관찰 포인트**
- Embedding을 지나면 `(1, T, C)`가 되고, **Block을 몇 개 지나도 `(1, T, C)` 그대로**입니다. 그래서 Block을 원하는 만큼 쌓을 수 있어요. (Day 15)
- shape가 잠깐 바뀌는 곳은 딱 두 군데: Attention 안의 `(B, n_head, T, T)`와 FFN 안의 `(B, T, 4C)`
- 마지막에 `lm_head`가 `C → vocab_size`로 바꿔서 **글자마다 점수**가 나옵니다.

## 3. 최종 흐름에 Day 붙이기

커리큘럼 학습 원칙 5번의 최종 흐름입니다. 이제 각 화살표가 **코드 몇 줄**인지 알고 있어요.

```text
문장                      "나는 오늘 학교에 간다"
 ↓  Tokenizer             encode()                         Day 8
Token ID                  (B, T)
 ↓  Embedding             tok_emb(idx)                     Day 9
 ↓  Position              + pos_emb(arange(T))             Day 10
                          (B, T, C)
 ↓  Self-Attention        x + attn(ln1(x))                 Day 11~14, 16
 ↓  FFN                   x + mlp(ln2(x))                  Day 4, 17, 16
 ↓  Transformer Block × N self.blocks                      Day 15
                          (B, T, C)
 ↓  Linear                lm_head(ln_f(x))                 Day 3, 19
                          (B, T, vocab_size)
 ↓  Softmax               softmax(logits[:, -1, :])        Day 5
 ↓  Next Token            multinomial(probs)               Day 5, 20
 ↓  반복 생성              torch.cat → 다시 모델에           Day 20
```

## 4. 공식은 코드 어디에 있나?

시험 볼 때 외웠던 공식이 실제 코드에서는 **한 줄**입니다. 코드에서 직접 찾아 출력해봅시다.

In [ ]:
def find_line(fn, keyword):
    try:
        for line in inspect.getsource(fn).splitlines():
            if keyword in line:
                return line.split("#")[0].strip()
    except (OSError, TypeError):
        return "(소스를 찾지 못했어요)"
    return "(없음)"

formula_map = [
    ("y = xW + b",                "nn.Linear 전부 (c_attn, c_proj, MLP, lm_head)", "self.c_attn = nn.Linear", CausalSelfAttention.__init__, "Day 3"),
    ("QK^T / √dk",                "CausalSelfAttention.forward",                   "math.sqrt",               CausalSelfAttention.forward,  "Day 12·13"),
    ("Causal Mask",               "CausalSelfAttention.forward",                   "masked_fill",             CausalSelfAttention.forward,  "Day 14"),
    ("softmax(...)",              "CausalSelfAttention.forward",                   "F.softmax",               CausalSelfAttention.forward,  "Day 5·11"),
    ("× V",                       "CausalSelfAttention.forward",                   "att @ v",                 CausalSelfAttention.forward,  "Day 12"),
    ("GELU",                      "MLP.__init__",                                  "GELU",                    MLP.__init__,                 "Day 4"),
    ("LayerNorm + Residual",      "Block.forward",                                 "self.ln1",                Block.forward,                "Day 15·16"),
    ("Embedding + Position",      "GPT.forward",                                   "x = tok + pos",           GPT.forward,                  "Day 9·10"),
    ("Cross Entropy",             "GPT.forward",                                   "cross_entropy",           GPT.forward,                  "Day 6"),
    ("다음 Token 뽑기",            "GPT.generate",                                  "multinomial",             GPT.generate,                 "Day 5·20"),
]

for name, where, kw, fn, day in formula_map:
    print(f"■ {name}  ({day})  →  {where}")
    print(f"    {find_line(fn, kw)}")

> 💡 `softmax(QKᵀ/√dk)V`는 코드에서 **세 줄**(`att = ... / math.sqrt(hs)` → `F.softmax` → `att @ v`)로 나뉘어 있고,
> 그 사이에 Causal Mask 한 줄이 끼어 있습니다. Day 13에서 그린 단계 다이어그램과 똑같은 순서예요.

LayerNorm 공식(평균을 빼고 표준편차로 나누기)은 `nn.LayerNorm` 안에 숨어 있습니다. 직접 확인해볼까요?

In [ ]:
ln = model.blocks[0].ln1
x_test = torch.randn(1, 3, C)

mean = x_test.mean(dim=-1, keepdim=True)
var = x_test.var(dim=-1, keepdim=True, unbiased=False)
manual = (x_test - mean) / torch.sqrt(var + ln.eps) * ln.weight + ln.bias   # Day 16 공식

print("nn.LayerNorm == 직접 계산 ?", torch.allclose(ln(x_test), manual, atol=1e-6))

## 5. 3주 동안 배운 것이 GPT에서 하는 일

| 주차 | 배운 것 | GPT 안에서 하는 일 |
|---|---|---|
| **1주차** | 행렬 곱, Linear (Day 2·3) | Q/K/V 만들기, FFN, 점수 계산 등 **거의 모든 계산** |
| | Activation / GELU (Day 4) | FFN 안에서 비선형성 → 복잡한 패턴 표현 |
| | Softmax (Day 5) | Attention Weight 만들기, 다음 Token 확률 |
| | Loss / Cross Entropy (Day 6) | "다음 글자를 얼마나 잘 맞혔나" 채점 |
| | Backprop / Optimizer (Day 7) | 채점 결과로 모든 Weight를 조금씩 고치기 |
| **2주차** | Tokenizer (Day 8) | 문장 ↔ 숫자 |
| | Embedding (Day 9) | Token ID → 의미를 담은 벡터 |
| | Position (Day 10) | 순서 정보 넣기 |
| | Self-Attention, Q/K/V (Day 11·12) | Token끼리 정보 주고받기 |
| | √dk (Day 13) | Softmax가 한쪽으로 쏠리지 않게 |
| | Causal Mask, Multi-Head (Day 14) | 미래 커닝 방지, 여러 관계를 동시에 보기 |
| **3주차** | Block, Pre-LN (Day 15) | Attention + FFN 묶음을 N번 쌓기 |
| | Residual, LayerNorm (Day 16) | 깊게 쌓아도 학습이 잘 되게 |
| | FFN (Day 17) | Token마다 따로 "생각"하기 |
| | Encoder / Decoder (Day 18) | GPT = Decoder만 쓴 구조 |
| | GPT 구조 (Day 19) | 위 부품 전체를 조립 |
| | Next Token Prediction (Day 20) | 학습 목표와 생성 방식 |

## 6. 맛보기 학습: Loss가 정말 내려갈까?

학습 데이터는 Day 20에서 배운 대로 **한 칸씩 밀린 x, y**입니다.
Loss는 처음에 Day 6에서 배운 `log(vocab_size)` 근처에서 시작해야 해요. (아무것도 모르는 모델 = 균등 분포)

> ⚠️ 이건 "조립한 코드가 돌아간다"를 확인하는 **맛보기**입니다.
> 4주차에서 이 모든 부품을 처음부터 직접 만들고, 더 큰 데이터로 제대로 학습시킵니다.

In [ ]:
torch.manual_seed(42)
model = GPT(vocab_size, C, n_head, n_layer, block_size)

def get_batch(batch_size=16):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])          # Day 20: 입력
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])  # Day 20: 한 칸 밀린 정답
    return x, y

xb, yb = get_batch(1)
print("x:", repr(decode(xb[0].tolist())))
print("y:", repr(decode(yb[0].tolist())))

start = torch.tensor([encode("나는")])
torch.manual_seed(0)
before = decode(model.generate(start, 40)[0].tolist())

_, loss0 = model(*get_batch())
print()
print(f"학습 전 Loss  : {loss0.item():.3f}")
print(f"log(vocab_size): {math.log(vocab_size):.3f}   ← Day 6: 랜덤 모델의 기대 Loss")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)   # Day 7: GPT가 쓰는 Optimizer
losses = []

for step in range(400):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)        # Day 19·20: forward + Cross Entropy
    optimizer.zero_grad()               # Day 7
    loss.backward()                     # Day 7: Backpropagation
    optimizer.step()                    # Day 7: Weight Update
    losses.append(loss.item())
    if step % 100 == 0 or step == 399:
        print(f"step {step:3d} | loss {loss.item():.3f}")

In [ ]:
plt.figure(figsize=(8, 3.8))
plt.plot(losses, linewidth=1.5, label="학습 Loss")
plt.axhline(math.log(vocab_size), color="gray", linestyle="--", label=f"log(vocab_size) = {math.log(vocab_size):.2f}")
plt.xlabel("step")
plt.ylabel("Loss")
plt.title("mini GPT 맛보기 학습", fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
torch.manual_seed(0)
after = decode(model.generate(start, 40)[0].tolist())

print("학습 전:", before)
print("학습 후:", after)

학습 전에는 아무 글자나 이어 붙이지만, 400 step만 학습해도 **학습 데이터에 있던 문장 비슷한 것**을 만들어냅니다.
데이터가 아주 작아서 사실상 외운 것에 가깝지만, **다음 글자 예측만으로 문장이 만들어진다**는 GPT의 원리는 그대로예요.

마지막으로 학습된 모델의 Attention이 어디를 보는지 그려봅시다. (Block 0, Head 0)

In [ ]:
with torch.no_grad():
    ids = torch.tensor([encode(sentence)])
    T = ids.shape[1]
    x = model.tok_emb(ids) + model.pos_emb(torch.arange(T))
    blk = model.blocks[0]
    h = blk.ln1(x)
    q, k, _ = blk.attn.c_attn(h).split(C, dim=2)
    hs = C // n_head
    q = q.view(1, T, n_head, hs).transpose(1, 2)
    k = k.view(1, T, n_head, hs).transpose(1, 2)
    att = q @ k.transpose(-2, -1) / math.sqrt(hs)
    att = att.masked_fill(blk.attn.mask[:T, :T] == 0, float("-inf"))
    att = F.softmax(att, dim=-1)[0, 0]

labels = [c if c != " " else "_" for c in sentence]   # 띄어쓰기는 _로 표시
plt.figure(figsize=(6, 5))
plt.imshow(att, cmap="Blues")
plt.xticks(range(T), labels)
plt.yticks(range(T), labels)
plt.xlabel("참고하는 글자 (Key)")
plt.ylabel("지금 글자 (Query)")
plt.title("학습 후 Attention Weight (Block 0, Head 0)", fontsize=12)
plt.colorbar()
plt.show()

위쪽 오른편 삼각형이 전부 비어 있죠? **Causal Mask** 때문에 미래 글자는 볼 수 없습니다. (Day 14)

## 7. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1. shape 추적 빈칸 채우기**

설정이 `B=4, T=16, C=64, n_head=8, vocab_size=100`일 때 각 단계의 shape를 예측하세요.
2절의 표를 참고하되, **숫자는 이 설정에 맞게** 바꿔야 합니다.

In [ ]:
my_shapes = {
    "head_size":              "?",   # TODO: 숫자 하나
    "Token Embedding":        "?",   # TODO: 튜플
    "q (head 나눈 뒤)":        "?",   # TODO
    "QK^T / √dk":             "?",   # TODO
    "FFN 확장 (×4)":           "?",   # TODO
    "logits":                 "?",   # TODO
}

# ---- 아래는 채점 코드 (실제 모델로 직접 측정) ----
_B, _T, _C, _nh, _V = 4, 16, 64, 8, 100
_m = GPT(_V, _C, _nh, 1, 32)
with torch.no_grad():
    _idx = torch.randint(_V, (_B, _T))
    _x = _m.tok_emb(_idx) + _m.pos_emb(torch.arange(_T))
    _blk = _m.blocks[0]
    _q, _k, _ = _blk.attn.c_attn(_blk.ln1(_x)).split(_C, dim=2)
    _hs = _C // _nh
    _q4 = _q.view(_B, _T, _nh, _hs).transpose(1, 2)
    _k4 = _k.view(_B, _T, _nh, _hs).transpose(1, 2)
    _truth = {
        "head_size":       _hs,
        "Token Embedding": tuple(_m.tok_emb(_idx).shape),
        "q (head 나눈 뒤)": tuple(_q4.shape),
        "QK^T / √dk":      tuple((_q4 @ _k4.transpose(-2, -1)).shape),
        "FFN 확장 (×4)":    tuple(_blk.mlp.net[0](_x).shape),
        "logits":          tuple(_m(_idx)[0].shape),
    }

ok = True
for key in _truth:
    mark = "✅" if my_shapes[key] == _truth[key] else "❌"
    ok &= mark == "✅"
    print(f"{mark} {key:18s} 내 답: {my_shapes[key]}")
assert ok, "❌ 표시된 항목을 다시 생각해보세요"
print("Q1 정답!")

**Q2. 파라미터 개수 세기**

설정 `vocab_size=50, C=32, n_head=4, n_layer=2, block_size=16`인 GPT의 **전체 파라미터 개수**를 손으로 계산하세요.

힌트 (1절 코드 기준):
- `tok_emb`: `vocab_size × C`, `pos_emb`: `block_size × C`
- Block 1개: `ln1`(2C) + `c_attn`(C×3C + 3C) + `c_proj`(C×C + C) + `ln2`(2C) + MLP 두 Linear(C×4C + 4C, 4C×C + C)
- `ln_f`: 2C, `lm_head`: `C × vocab_size` (bias 없음)
- `n_head`는 파라미터 개수에 영향을 줄까요? (Day 14)

In [ ]:
my_count = "?"   # TODO: 숫자 하나

_m2 = GPT(vocab_size=50, C=32, n_head=4, n_layer=2, block_size=16)
assert my_count == sum(p.numel() for p in _m2.parameters()), "다시 세어보세요. 부품별로 나눠서 계산해보면 좋아요"
print("Q2 정답!")

**Q3. 버그 찾기**

아래 `BuggyBlock`에는 버그가 **하나** 있습니다. 진단 함수 두 개를 직접 실행해서 무엇이 잘못됐는지 찾으세요.
- `sees_future(block)`: 미래 Token을 바꿨을 때 과거 위치 출력이 바뀌면 `True` (커닝 중!)
- `has_residual(block)`: Attention·FFN의 출력을 0으로 만들었을 때 입력이 그대로 통과하면 `True`

보기:
- `"A"`: Residual Connection이 빠졌다
- `"B"`: Causal Mask가 빠졌다
- `"C"`: 둘 다 정상이다

In [ ]:
import copy

class BuggyAttention(CausalSelfAttention):
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)
        hs = C // self.n_head
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        att = q @ k.transpose(-2, -1) / math.sqrt(hs)
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class BuggyBlock(Block):
    def __init__(self, C, n_head, block_size):
        super().__init__(C, n_head, block_size)
        self.attn = BuggyAttention(C, n_head, block_size)

def sees_future(block, T=6, C=32):
    torch.manual_seed(0)
    x = torch.randn(1, T, C)
    x2 = x.clone()
    x2[0, -1] = torch.randn(C)                 # 마지막(미래) Token만 바꾸기
    with torch.no_grad():
        out1, out2 = block(x), block(x2)
    return not torch.allclose(out1[0, :-1], out2[0, :-1], atol=1e-6)

def has_residual(block, T=6, C=32):
    b = copy.deepcopy(block)
    with torch.no_grad():
        for lin in [b.attn.c_proj, b.mlp.net[2]]:  # 두 경로의 마지막 Linear를 0으로
            lin.weight.zero_()
            lin.bias.zero_()
        x = torch.randn(1, T, C)
        return torch.allclose(b(x), x)

buggy = BuggyBlock(32, 4, 16)

# TODO: 아래 두 줄의 주석을 풀고 실행해서 결과를 보세요
# print("sees_future :", sees_future(buggy))
# print("has_residual:", has_residual(buggy))

my_answer = "?"   # TODO: "A", "B", "C" 중 하나

In [ ]:
_flags = {
    "A": not has_residual(buggy),
    "B": sees_future(buggy),
    "C": has_residual(buggy) and not sees_future(buggy),
}
assert my_answer in _flags, '"A", "B", "C" 중 하나로 답해주세요'
assert _flags[my_answer], "진단 함수 결과를 다시 확인해보세요"
print("Q3 정답!")

**Q4. Block의 forward 직접 쓰기**

`MyBlock`의 `forward`를 완성하세요. 1절 `Block`을 보지 말고 먼저 떠올려보세요.
- **Pre-LN**: LayerNorm을 Attention / FFN **앞에** 적용 (Day 15)
- **Residual**: 결과를 입력에 **더하기** (Day 16)
- 두 줄이면 됩니다

In [ ]:
class MyBlock(nn.Module):
    def __init__(self, C, n_head, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(C)
        self.attn = CausalSelfAttention(C, n_head, block_size)
        self.ln2 = nn.LayerNorm(C)
        self.mlp = MLP(C)

    def forward(self, x):
        # TODO: 두 줄을 채우고 아래 return None을 지우세요
        return None

_ref = Block(32, 4, 16)
_mine = MyBlock(32, 4, 16)
_mine.load_state_dict(_ref.state_dict())     # 같은 Weight로 맞추기

_x = torch.randn(2, 10, 32)
_out = _mine(_x)
assert _out is not None, "forward를 완성해주세요"
assert torch.allclose(_out, _ref(_x), atol=1e-6), "Pre-LN과 Residual의 위치를 다시 생각해보세요"
print("Q4 정답!")

## 8. 최종 점검: 커리큘럼 최종 질문 전체

커리큘럼에서 "최종적으로 설명할 수 있어야 하는 질문"입니다. **코드 없이 말로** 답해보세요.
막히는 질문은 옆에 적힌 Day 노트북을 다시 보면 됩니다.

### 딥러닝
| 질문 | 다시 볼 Day |
|---|---|
| Weight는 무엇인가? | Day 3 |
| Bias는 왜 필요한가? | Day 3 |
| Linear Layer는 어떤 역할을 하는가? | Day 3 |
| Activation Function은 왜 필요한가? | Day 4 |
| Softmax는 왜 사용하는가? | Day 5 |
| Loss는 무엇인가? | Day 6 |
| Backpropagation은 왜 필요한가? | Day 7 |
| Optimizer는 무엇을 하는가? | Day 7 |

### NLP
| 질문 | 다시 볼 Day |
|---|---|
| Token은 무엇인가? | Day 8 |
| Tokenizer는 왜 필요한가? | Day 8 |
| Token ID와 Embedding의 차이는 무엇인가? | Day 8, 9 |
| Embedding은 무엇인가? | Day 9 |
| Position 정보는 왜 필요한가? | Day 10 |

### Attention
| 질문 | 다시 볼 Day |
|---|---|
| Self-Attention은 무엇인가? | Day 11 |
| Query / Key / Value는 무엇인가? | Day 12 |
| QKᵀ는 왜 계산하는가? | Day 2, 12 |
| Softmax는 Attention에서 왜 사용하는가? | Day 5, 11, 13 |
| V는 어떤 역할을 하는가? | Day 12 |
| Multi-Head Attention은 왜 필요한가? | Day 14 |
| Causal Mask는 왜 필요한가? | Day 14 |

### Transformer
| 질문 | 다시 볼 Day |
|---|---|
| Transformer Block에는 무엇이 들어가는가? | Day 15 |
| Residual Connection은 왜 사용하는가? | Day 16 |
| LayerNorm은 왜 필요한가? | Day 16 |
| FFN은 Attention과 어떤 차이가 있는가? | Day 11, 17 |

### GPT
| 질문 | 다시 볼 Day |
|---|---|
| GPT는 Transformer와 어떤 관계인가? | Day 18, 19 |
| GPT는 왜 다음 Token을 예측하는가? | Day 20 |
| Training과 Inference는 무엇이 다른가? | Day 20 |
| GPT는 문장을 어떻게 생성하는가? | Day 20 |

### 마지막 질문 (학습 원칙 5)
> 지금 입력 데이터가 모델 내부에서 어떤 과정을 거쳐 출력으로 변환되는가?

3절의 흐름을 **보지 않고** 처음부터 끝까지 말로 설명할 수 있으면 3주차 목표 달성입니다. 🎉

**내 답:**

*딥러닝*
- Weight:
- Bias:
- Linear Layer:
- Activation Function:
- Softmax:
- Loss:
- Backpropagation:
- Optimizer:

*NLP*
- Token:
- Tokenizer:
- Token ID vs Embedding:
- Embedding:
- Position:

*Attention*
- Self-Attention:
- Query / Key / Value:
- QKᵀ:
- Attention의 Softmax:
- V의 역할:
- Multi-Head:
- Causal Mask:

*Transformer*
- Block 구성:
- Residual:
- LayerNorm:
- FFN vs Attention:

*GPT*
- GPT와 Transformer:
- 다음 Token 예측:
- Training vs Inference:
- 문장 생성:

*마지막 질문*
-